# E1 v5.3 — Standalone Held-out Test

Notebook này chỉ đánh giá **E1 MobileNetV3-Small đã freeze**.

Nó không train, không fine-tune, không recalibrate threshold, không resample Test và không chạy lại SCRFD.

Bắt buộc reuse:

```text
E1 split manifest
E1 SCRFD bbox cache
E1 frozen ONNX
E1 Validation-calibrated threshold
```

PAD score:

```text
d = real_logit - logsumexp([physical_spoof_logit, digital_spoof_logit])
REAL iff d >= locked_validation_logit_threshold
```

Kết quả được lưu thành JSON/CSV/PNG + ZIP để archive khỏi Kaggle.


In [ ]:
!pip install -q onnxruntime-gpu scikit-learn pandas matplotlib

import os, glob, json, time, hashlib, zipfile, sys
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import onnxruntime as ort

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)
print("ORT:", ort.__version__)
print("Providers:", ort.get_available_providers())


## Kaggle inputs

Attach:

1. the same CelebA-Spoof dataset used by E1;
2. E1 artifacts.

Required:

```text
celeba_scrfd_bbox_cache_v5_3_<mode>_seed42.json
celeba_spoof_<mode>_v5_3_edge_mnv3_small_seed42.npz
mnv3s_e1_<mode>_v5_3_edge_best.onnx
```

Threshold metadata — at least one:

```text
mnv3s_e1_<mode>_v5_3_edge_runtime_config.json
mnv3s_e1_<mode>_v5_3_edge_best_meta.json
```

Recommended:

```text
mnv3s_e1_<mode>_v5_3_edge_run_config.json
```


In [ ]:
E1_RUN_MODE = "preliminary"  # "preliminary" or "official"
E1_ARTIFACT_DIR = ""         # blank = exact-name discovery under /kaggle/input

DATA_ROOT = (
    "/kaggle/input/datasets/attentionlayer241/"
    "celeba-spoof-for-face-antispoofing/"
    "CelebA_Spoof_/CelebA_Spoof"
)

MODEL_PATH_OVERRIDE = ""
CACHE_PATH_OVERRIDE = ""
MANIFEST_PATH_OVERRIDE = ""
RUNTIME_CONFIG_OVERRIDE = ""
BEST_META_OVERRIDE = ""
RUN_CONFIG_OVERRIDE = ""

# Last-resort only: exact locked Validation threshold.
MANUAL_LOCKED_LOGIT_THRESHOLD = None

BATCH_SIZE = 128
NUM_WORKERS = 4
OUTPUT_DIR = f"e1_{E1_RUN_MODE}_heldout_test_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def discover_exact(filename, artifact_dir="", required=True):
    if artifact_dir:
        p = os.path.join(artifact_dir, filename)
        if os.path.exists(p):
            return p
        if required:
            raise FileNotFoundError(p)
        return None
    matches = sorted(set(glob.glob(os.path.join("/kaggle/input","**",filename), recursive=True)))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple copies of {filename} found:\n" + "\n".join(matches) +
            "\nSet E1_ARTIFACT_DIR or an explicit override."
        )
    if required:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input.")
    return None

def resolve(override, filename, required=True):
    if override:
        if not os.path.exists(override):
            raise FileNotFoundError(override)
        return override
    return discover_exact(filename, E1_ARTIFACT_DIR, required)

CACHE_NAME = f"celeba_scrfd_bbox_cache_v5_3_{E1_RUN_MODE}_seed42.json"
MANIFEST_NAME = f"celeba_spoof_{E1_RUN_MODE}_v5_3_edge_mnv3_small_seed42.npz"
MODEL_NAME = f"mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_best.onnx"
RUNTIME_NAME = f"mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_runtime_config.json"
BEST_META_NAME = f"mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_best_meta.json"
RUN_CONFIG_NAME = f"mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_run_config.json"

CACHE_PATH = resolve(CACHE_PATH_OVERRIDE, CACHE_NAME)
MANIFEST_PATH = resolve(MANIFEST_PATH_OVERRIDE, MANIFEST_NAME)
MODEL_PATH = resolve(MODEL_PATH_OVERRIDE, MODEL_NAME)
RUNTIME_CONFIG_PATH = resolve(RUNTIME_CONFIG_OVERRIDE, RUNTIME_NAME, required=False)
BEST_META_PATH = resolve(BEST_META_OVERRIDE, BEST_META_NAME, required=False)
RUN_CONFIG_PATH = resolve(RUN_CONFIG_OVERRIDE, RUN_CONFIG_NAME, required=False)

TRAIN_JSON = os.path.join(DATA_ROOT, "metas/intra_test/train_label.json")
TEST_JSON = os.path.join(DATA_ROOT, "metas/intra_test/test_label.json")
for p in [TRAIN_JSON, TEST_JSON]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)

print("Model:", MODEL_PATH)
print("Cache:", CACHE_PATH)
print("Manifest:", MANIFEST_PATH)
print("Runtime config:", RUNTIME_CONFIG_PATH)
print("Best meta:", BEST_META_PATH)
print("Run config:", RUN_CONFIG_PATH)


In [ ]:
CELEBA_MEAN = [0.5931, 0.4690, 0.4229]
CELEBA_STD = [0.2471, 0.2214, 0.2157]
INPUT_SIZE = 224

with open(TRAIN_JSON) as f:
    train_meta = json.load(f)
with open(TEST_JSON) as f:
    test_meta = json.load(f)

manifest = np.load(MANIFEST_PATH, allow_pickle=False)
for name in ["train_keys","val_keys","test_keys"]:
    if name not in manifest.files:
        raise RuntimeError(f"Manifest missing {name}")

train_keys = manifest["train_keys"].astype(str)
val_keys = manifest["val_keys"].astype(str)
test_keys = manifest["test_keys"].astype(str)

with open(CACHE_PATH) as f:
    cache_payload = json.load(f)
if "records" not in cache_payload:
    raise RuntimeError("Invalid SCRFD cache.")
bbox_cache = cache_payload["records"]
cache_policy = cache_payload.get("policy", {})

if abs(float(cache_policy.get("scrfd_crop_factor",-1)) - 1.55) > 1e-9:
    raise RuntimeError(f"Unexpected SCRFD crop policy: {cache_policy}")
if abs(float(cache_policy.get("celeba_fallback_crop_factor",-1)) - 1.50) > 1e-9:
    raise RuntimeError(f"Unexpected fallback crop policy: {cache_policy}")

missing = [k for k in test_keys if k not in bbox_cache]
if missing:
    raise RuntimeError(f"{len(missing)} Test keys missing from cache.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def split_fp(keys):
    return hashlib.sha256("\n".join(sorted(map(str,keys))).encode()).hexdigest()

split_fingerprints = {
    "train": split_fp(train_keys),
    "val": split_fp(val_keys),
    "test": split_fp(test_keys),
}

threshold_sources = []
if RUNTIME_CONFIG_PATH:
    with open(RUNTIME_CONFIG_PATH) as f:
        cfg = json.load(f)
    if "calibrated_logit_threshold" in cfg:
        threshold_sources.append(("runtime_config", float(cfg["calibrated_logit_threshold"])))
if BEST_META_PATH:
    with open(BEST_META_PATH) as f:
        meta = json.load(f)
    if "calibrated_logit_threshold" in meta:
        threshold_sources.append(("best_meta", float(meta["calibrated_logit_threshold"])))
if MANUAL_LOCKED_LOGIT_THRESHOLD is not None:
    threshold_sources.append(("manual", float(MANUAL_LOCKED_LOGIT_THRESHOLD)))

if not threshold_sources:
    raise RuntimeError("No locked Validation threshold found.")
vals = [v for _,v in threshold_sources]
if max(vals)-min(vals) > 1e-8:
    raise RuntimeError(f"Conflicting threshold artifacts: {threshold_sources}")

LOCKED_LOGIT_THRESHOLD = vals[0]
LOCKED_PROBABILITY_THRESHOLD = float(1/(1+np.exp(-LOCKED_LOGIT_THRESHOLD)))

if RUN_CONFIG_PATH:
    with open(RUN_CONFIG_PATH) as f:
        rcfg = json.load(f)
    expected = rcfg.get("split_fingerprints")
    if expected:
        for s in ["train","val","test"]:
            if expected.get(s) != split_fingerprints[s]:
                raise RuntimeError(f"{s} fingerprint differs from E1 run config.")

print("=== E1 TEST PROTOCOL VERIFIED ===")
print("Test N:", len(test_keys))
print("Locked d threshold:", LOCKED_LOGIT_THRESHOLD)
print("Locked p threshold:", LOCKED_PROBABILITY_THRESHOLD)
print("Threshold sources:", threshold_sources)
print("Test fingerprint:", split_fingerprints["test"])
print("Model SHA256:", sha256_file(MODEL_PATH))


In [ ]:
def label_of(meta):
    raw = int(meta[40])
    if raw == 0:
        return 0
    if raw in [1,2,3,4,5,6,7]:
        return 1
    return 2

CLASS_NAMES = {0:"Real",1:"Physical Spoof",2:"Digital Spoof"}

def subject_id(path):
    parts = path.replace("\\","/").split("/")
    for i,p in enumerate(parts[:-1]):
        if p.lower() in {"train","test"}:
            return parts[i+1]
    return "unknown"

def runtime_crop_bgr(img, bbox, factor):
    H,W = img.shape[:2]
    x1,y1,x2,y2 = map(float,bbox)
    w,h = x2-x1,y2-y1
    if w <= 0 or h <= 0:
        raise ValueError(bbox)
    m = max(w,h)
    cx,cy = x1+w/2, y1+h/2
    xs = int(cx-m*factor/2)
    ys = int(cy-m*factor/2)
    size = int(m*factor)

    ix1,iy1 = max(0,xs),max(0,ys)
    ix2,iy2 = min(W,xs+size),min(H,ys+size)
    crop = img[iy1:iy2,ix1:ix2]
    if crop.size == 0:
        raise RuntimeError("Empty crop")

    out = cv2.copyMakeBorder(
        crop,
        max(0,-ys), max(0,ys+size-H),
        max(0,-xs), max(0,xs+size-W),
        cv2.BORDER_REFLECT_101
    )
    if out.shape[:2] != (size,size):
        out = cv2.resize(out,(size,size),interpolation=cv2.INTER_AREA)
    return out

def preprocess_e1(key):
    img = cv2.imread(os.path.join(DATA_ROOT,key), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(key)
    rec = bbox_cache[key]
    crop = runtime_crop_bgr(img, rec["bbox_xyxy"], float(rec["crop_factor"]))
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    h,w = rgb.shape[:2]
    ratio = INPUT_SIZE/max(h,w)
    sh,sw = max(1,int(h*ratio)),max(1,int(w*ratio))
    interp = cv2.INTER_LANCZOS4 if ratio>1 else cv2.INTER_AREA
    x = cv2.resize(rgb,(sw,sh),interpolation=interp)
    dh,dw = INPUT_SIZE-sh, INPUT_SIZE-sw
    x = cv2.copyMakeBorder(
        x,dh//2,dh-dh//2,dw//2,dw-dw//2,cv2.BORDER_REFLECT_101
    )

    x = x.transpose(2,0,1).astype(np.float32)/255.0
    mean = np.asarray(CELEBA_MEAN,dtype=np.float32)[:,None,None]
    std = np.asarray(CELEBA_STD,dtype=np.float32)[:,None,None]
    return np.ascontiguousarray((x-mean)/std)

class E1TestDataset(Dataset):
    def __init__(self, keys):
        self.keys = list(map(str,keys))
    def __len__(self):
        return len(self.keys)
    def __getitem__(self,idx):
        key = self.keys[idx]
        meta = test_meta[key]
        rec = bbox_cache[key]
        return (
            torch.from_numpy(preprocess_e1(key)),
            torch.tensor(label_of(meta),dtype=torch.long),
            key,
            torch.tensor(int(meta[40]),dtype=torch.long),
            rec.get("bbox_source","unknown"),
        )

test_loader = DataLoader(
    E1TestDataset(test_keys),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)
print("Test loader ready:", len(test_keys))


In [ ]:
def pad_score(logits):
    logits = np.asarray(logits,dtype=np.float64)
    real = logits[:,0]
    spoof = logits[:,1:]
    m = np.max(spoof,axis=1)
    lse = m + np.log(np.exp(spoof[:,0]-m)+np.exp(spoof[:,1]-m))
    return real-lse

def metrics_from(y3, scores, threshold):
    y3 = np.asarray(y3,dtype=np.int64)
    scores = np.asarray(scores,dtype=np.float64)
    real = y3==0
    attack = ~real
    pred_real = scores>=threshold
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    return {
        "Accuracy": float(np.mean(pred_real==real)),
        "APCER": apcer,
        "BPCER": bpcer,
        "ACER": 0.5*(apcer+bpcer),
        "AUC": float(roc_auc_score(real.astype(np.int64),scores)),
        "binary_confusion_matrix": confusion_matrix(
            real.astype(np.int64),pred_real.astype(np.int64),labels=[0,1]
        ).tolist()
    }


In [ ]:
providers = (
    ["CUDAExecutionProvider","CPUExecutionProvider"]
    if "CUDAExecutionProvider" in ort.get_available_providers()
    else ["CPUExecutionProvider"]
)
session = ort.InferenceSession(MODEL_PATH, providers=providers)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print("ORT providers:", session.get_providers())
print("Input:", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

rows=[]
all_logits=[]
all_y=[]
infer_s=0.0
t_eval=time.perf_counter()

for x,y,keys,attack_codes,bbox_sources in tqdm(test_loader):
    xnp=x.numpy().astype(np.float32,copy=False)
    t0=time.perf_counter()
    logits=session.run([output_name],{input_name:xnp})[0]
    infer_s += time.perf_counter()-t0
    if logits.ndim!=2 or logits.shape[1]!=3:
        raise RuntimeError(logits.shape)

    scores=pad_score(logits)
    p_real=1/(1+np.exp(-scores))
    pred_real=scores>=LOCKED_LOGIT_THRESHOLD
    pred3=logits.argmax(1)
    ynp=y.numpy()
    ac=attack_codes.numpy()

    all_logits.append(logits)
    all_y.append(ynp)

    for i,key in enumerate(keys):
        rows.append({
            "key":key,
            "subject_id":subject_id(key),
            "attack_code":int(ac[i]),
            "true_class":int(ynp[i]),
            "true_class_name":CLASS_NAMES[int(ynp[i])],
            "bbox_source":bbox_sources[i],
            "logit_real":float(logits[i,0]),
            "logit_physical":float(logits[i,1]),
            "logit_digital":float(logits[i,2]),
            "pad_logit_score":float(scores[i]),
            "p_real":float(p_real[i]),
            "pred_real":bool(pred_real[i]),
            "pred_class_3":int(pred3[i]),
            "correct_binary":bool(pred_real[i]==(ynp[i]==0)),
            "correct_3class":bool(pred3[i]==ynp[i]),
        })

eval_s=time.perf_counter()-t_eval
pred_df=pd.DataFrame(rows)
logits_all=np.concatenate(all_logits)
y_all=np.concatenate(all_y)
scores_all=pad_score(logits_all)

metrics=metrics_from(y_all,scores_all,LOCKED_LOGIT_THRESHOLD)
metrics["Accuracy3Class"]=float(np.mean(logits_all.argmax(1)==y_all))
metrics["N"]=int(len(y_all))
metrics["locked_logit_threshold"]=LOCKED_LOGIT_THRESHOLD
metrics["locked_probability_threshold"]=LOCKED_PROBABILITY_THRESHOLD
metrics["model_inference_seconds"]=float(infer_s)
metrics["total_evaluation_seconds"]=float(eval_s)
metrics["model_inference_ms_per_image"]=float(infer_s*1000/len(y_all))

print("\n=== E1 HELD-OUT TEST ===")
for k in ["Accuracy","Accuracy3Class","APCER","BPCER","ACER","AUC"]:
    print(f"{k:16s}: {100*metrics[k]:.4f}%")
print("Inference ms/image:",metrics["model_inference_ms_per_image"])


In [ ]:
pred_path=os.path.join(OUTPUT_DIR,"e1_test_predictions.csv")
pred_df.to_csv(pred_path,index=False)

breakdown=[]
for attack_code,g in pred_df.groupby("attack_code",sort=True):
    c=int(g["true_class"].iloc[0])
    if c==0:
        metric_name="BPCER"
        error=float(np.mean(~g["pred_real"]))
    else:
        metric_name="APCER"
        error=float(np.mean(g["pred_real"]))
    breakdown.append({
        "attack_code":int(attack_code),
        "class":CLASS_NAMES[c],
        "N":int(len(g)),
        "mean_d":float(g["pad_logit_score"].mean()),
        "median_d":float(g["pad_logit_score"].median()),
        "std_d":float(g["pad_logit_score"].std(ddof=0)),
        "error_metric":metric_name,
        "error_rate":error,
    })
breakdown_df=pd.DataFrame(breakdown)
breakdown_path=os.path.join(OUTPUT_DIR,"e1_attack_breakdown.csv")
breakdown_df.to_csv(breakdown_path,index=False)

cm=np.asarray(metrics["binary_confusion_matrix"])
fig,ax=plt.subplots(figsize=(5,5))
ConfusionMatrixDisplay(cm,display_labels=["Attack","Real"]).plot(ax=ax,values_format="d")
ax.set_title("E1 Held-out Test — Binary PAD")
fig.tight_layout()
cm_path=os.path.join(OUTPUT_DIR,"e1_binary_confusion_matrix.png")
fig.savefig(cm_path,dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(8,5))
for cid in [0,1,2]:
    vals=pred_df.loc[pred_df["true_class"]==cid,"pad_logit_score"].to_numpy()
    ax.hist(vals,bins=80,alpha=0.45,density=True,label=CLASS_NAMES[cid])
ax.axvline(LOCKED_LOGIT_THRESHOLD,linestyle="--",label="Locked Validation threshold")
ax.set_xlabel("PAD logit score d")
ax.set_ylabel("Density")
ax.set_title("E1 Held-out Test — Score Distribution")
ax.legend()
fig.tight_layout()
score_path=os.path.join(OUTPUT_DIR,"e1_score_distribution.png")
fig.savefig(score_path,dpi=180)
plt.show()

summary={
    "experiment":"E1",
    "model":"MobileNetV3-Small spatial-only",
    "run_mode":E1_RUN_MODE,
    "model_file":os.path.basename(MODEL_PATH),
    "model_sha256":sha256_file(MODEL_PATH),
    "cache_file":os.path.basename(CACHE_PATH),
    "manifest_file":os.path.basename(MANIFEST_PATH),
    "split_fingerprints":split_fingerprints,
    "threshold_sources":threshold_sources,
    "metrics":metrics,
}
summary_path=os.path.join(OUTPUT_DIR,"e1_heldout_test_summary.json")
with open(summary_path,"w") as f:
    json.dump(summary,f,indent=2)

protocol={
    "run_mode":E1_RUN_MODE,
    "input_size":INPUT_SIZE,
    "rgb_mean":CELEBA_MEAN,
    "rgb_std":CELEBA_STD,
    "gamma":False,
    "scrfd_crop_factor":1.55,
    "celeba_fallback_crop_factor":1.50,
    "threshold_policy":"locked from Validation; never calibrated on Test",
    "locked_logit_threshold":LOCKED_LOGIT_THRESHOLD,
    "locked_probability_threshold":LOCKED_PROBABILITY_THRESHOLD,
    "test_fingerprint":split_fingerprints["test"],
}
with open(os.path.join(OUTPUT_DIR,"e1_test_protocol_snapshot.json"),"w") as f:
    json.dump(protocol,f,indent=2)

zip_path=f"{OUTPUT_DIR}.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for root,_,files in os.walk(OUTPUT_DIR):
        for name in files:
            p=os.path.join(root,name)
            z.write(p,arcname=os.path.relpath(p,OUTPUT_DIR))

print("Saved:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" ",name)
print("ZIP:",zip_path)
